In [ ]:
import lmstudio as lms
import json
from datetime import datetime

# Model laden (één keer)
model = lms.llm("google/gemma-4-e4b")

# ─────────────────────────────────────────
# STAP 2 – Regelgebaseerd: tijdslots
# ─────────────────────────────────────────
def bereken_tijdslots(agenda_info: str) -> list:
    """Geen LLM nodig – puur logica of agenda API koppeling."""
    # Vervang dit door echte agenda-logica of Google Calendar API
    return [
        {"dag": "maandag", "uren": 3},
        {"dag": "dinsdag", "uren": 2},
        {"dag": "woensdag", "uren": 4},
    ]

# ─────────────────────────────────────────
# STAP 5 – Regelgebaseerd: uren berekenen
# ─────────────────────────────────────────
def bereken_studieuren(moeilijkheid_json: str, tijdslots: list) -> dict:
    """Geen LLM nodig – formule op basis van moeilijkheidsgraad."""
    moeilijkheid = json.loads(moeilijkheid_json)
    return {vak: graad * 2 for vak, graad in moeilijkheid.items()}

# ─────────────────────────────────────────
# TOOLS voor .act() – LM Studio roept deze aan
# ─────────────────────────────────────────
def sla_prioriteiten_op(prioriteiten_json: str) -> str:
    """Sla de geprioriteerde vakkenlijst op."""
    # Hier kun je wegschrijven naar een bestand of database
    print(f"\n📋 Prioriteiten opgeslagen: {prioriteiten_json}")
    return "Prioriteiten opgeslagen."

def sla_planning_op(planning_tekst: str) -> str:
    """Sla de gegenereerde planning op."""
    with open("planning.txt", "w", encoding="utf-8") as f:
        f.write(planning_tekst)
    print("\n📅 Planning weggeschreven naar planning.txt")
    return "Planning opgeslagen."

def stuur_herinnering(bericht: str, tijdstip: str) -> str:
    """Simuleer het sturen van een herinnering (koppel later aan notificatie API)."""
    print(f"\n🔔 Herinnering ingepland voor {tijdstip}: {bericht}")
    return f"Herinnering ingepland voor {tijdstip}."

# ─────────────────────────────────────────
# HOOFD-AGENT
# ─────────────────────────────────────────
def run_studie_agent():
    print("=== 📚 Slimme Studie-Agent (LM Studio) ===\n")

    # STAP 1 – Gebruikersinvoer
    gebruikersinput = input("Voer je vakken en deadlines in (vrije tekst):\n> ")

    # STAP 2 – Tijdslots (regelgebaseerd)
    tijdslots = bereken_tijdslots(agenda_info="")
    print(f"\n⏰ Beschikbare tijdslots: {tijdslots}")

    # STAP 3 t/m 6 + 8 – LLM doet het zware werk via .act()
    chat = lms.Chat("""Je bent een slimme studie-planner assistent.
Je helpt studenten met het plannen van hun studietijd op basis van vakken, 
deadlines, tijdslots en moeilijkheidsgraden.
Werk stap voor stap:
1. Analyseer de vakken en deadlines
2. Bepaal prioriteiten
3. Schat moeilijkheidsgraad per vak (1-5)
4. Genereer een concrete weekplanning
5. Plan herinneringen in via de tool
Gebruik altijd de beschikbare tools om resultaten op te slaan."""
    )

    prompt = f"""
De student heeft het volgende ingevoerd:
{gebruikersinput}

Beschikbare tijdslots deze week:
{json.dumps(tijdslots, ensure_ascii=False)}

Maak nu:
1. Een geprioriteerde vakkenlijst (gebruik sla_prioriteiten_op)
2. Een moeilijkheidsgraad per vak (JSON formaat)
3. Een concrete weekplanning (gebruik sla_planning_op)
4. Stel 2 herinneringen in (gebruik stuur_herinnering)
"""

    chat.add_user_message(prompt)

    print("\n🤖 Agent is bezig...\n")

    model.act(
        chat,
        tools=[sla_prioriteiten_op, sla_planning_op, stuur_herinnering],
        on_message=chat.append,
        on_prediction_fragment=lambda f, *_: print(f.content, end="", flush=True),
    )

    # STAP 9 & 10 – Feedback verwerken
    print("\n\n─────────────────────────────")
    feedback = input("\n📝 Geef je feedback op de planning (of druk Enter om te stoppen):\n> ")

    if feedback.strip():
        chat.add_user_message(
            f"De student geeft deze feedback: '{feedback}'\n"
            "Pas de planning aan op basis van deze feedback en sla opnieuw op."
        )
        print("\n🔄 Planning aanpassen...\n")
        model.act(
            chat,
            tools=[sla_prioriteiten_op, sla_planning_op, stuur_herinnering],
            on_message=chat.append,
            on_prediction_fragment=lambda f, *_: print(f.content, end="", flush=True),
        )

    print("\n\n✅ Klaar! Bekijk planning.txt voor je definitieve planning.")

if __name__ == "__main__":
    run_studie_agent()

=== 📚 Slimme Studie-Agent (LM Studio) ===


⏰ Beschikbare tijdslots: [{'dag': 'maandag', 'uren': 3}, {'dag': 'dinsdag', 'uren': 2}, {'dag': 'woensdag', 'uren': 4}]


TypeError: Chat.__init__() got an unexpected keyword argument 'system_prompt'